# ARTISTA axolotl Stage 57

Recompute the selected spatial-domain ARIs in Fig. 4a and display the
reference dEGC edge recovery counts in Fig. 4c. The default mode uses
the archived domain assignments. The STBSR panel used a historical
encoder partition, separate from the current joint model's reconstruction benchmark.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
repo = next((p for p in (here, *here.parents) if (p / "config.json").is_file()
             and (p / "STBSR" / "pipeline.py").is_file()), None)
if repo is None:
    raise FileNotFoundError("Start Jupyter from the STBSR repository or its notebooks directory.")
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from STBSR.reproduction import (
    compare_paper_ari, display_labels, evaluate_file, paper_domains, plot_domains,
    read_reference, score_paper_domains,
)

config = json.loads((repo / "config.json").read_text(encoding="utf-8"))
benchmark = repo / "benchmark"
archived_runs = pd.read_csv(benchmark / "stbsr_per_run.csv", dtype={"dataset": str})
mode = os.environ.get("STBSR_RUN_MODE", "paper")  # "paper" or "train"
if mode not in {"paper", "train"}:
    raise ValueError("Use STBSR_RUN_MODE=paper or STBSR_RUN_MODE=train.")
device = os.environ.get("STBSR_DEVICE", "cuda")
data_root = Path(os.environ.get("STBSR_DATA_ROOT", repo / "data"))
from STBSR.reproduction import stage57_boundary


## Fig. 4a selected partitions and Supplementary Data 1

In [ ]:
dataset = "AX_Stage57"
seed = 46  # Reconstruction benchmark example for the current joint model.
spot_size = 2

domains = paper_domains(dataset)
methods = ["STBSR", "DeepST", "stGRL", "SEDR", "DeepGFT", "STAGATE", "SEPAR"]
scores = pd.DataFrame([
    dict(method=m, ARI=score_paper_domains(domains, domains[m])[0])
    for m in methods
]).set_index("method")
assert len(domains) == 4410 and abs(scores.loc["STBSR", "ARI"] - 0.6371158977637179) < 1e-12
display(scores.round(4))
fig = plot_domains(
    domains, {f"{m} (ARI {scores.loc[m, 'ARI']:.4f})": domains[m] for m in methods},
    size=spot_size,
)
display(fig)
plt.close(fig)

run_scores = archived_runs.loc[archived_runs.dataset.eq(dataset)]
display(run_scores[["run", "seed", "ARI", "NMI", "BF", "BandAcc"]].round(4))
print(f"Stage 57 reconstruction benchmark ARI: {run_scores.ARI.mean():.4f} "
      f"(sample SD {run_scores.ARI.std():.4f})")

## Fig. 4c dEGC interface

In [ ]:
edges = stage57_boundary(domains, methods)
expected = pd.read_csv(benchmark / "fig04_degC_boundary.csv")[edges.columns]
pd.testing.assert_frame_equal(edges, expected)
edges["recovery_percent"] = 100 * edges["edges"] / edges["reference_crossing_edges"]
assert tuple(edges.loc[edges.method.eq("STBSR"), ["edges", "reference_crossing_edges"]].iloc[0]) == (207, 587)
display(edges[["method", "edges", "reference_crossing_edges", "recovery_percent"]].round(2))
fig, ax = plt.subplots(figsize=(5.5, 3.2))
colors = ["#355f91" if method == "STBSR" else "#a2a8ae" for method in edges.method]
ax.barh(edges.method, edges.recovery_percent, color=colors)
ax.invert_yaxis()
ax.set_xlabel("Reference dEGC edges recovered (%)")
fig.tight_layout()
display(fig)
plt.close(fig)

## Fig. 4b and d measured markers and reported GO terms

In [ ]:
gene_pairs = {
    "dEGC": ("GFAP", "VIM"),
    "wntEGC": ("WNT8B", "WNT2B"),
    "dNBL1": ("NOVA1", "SLC32A1"),
    "CP": ("MFRP", "KRT18"),
    "nptxEX": ("NPTX2", "TBR1"),
}
fig, axes = plt.subplots(2, 5, figsize=(11, 4.5))
for column, genes in enumerate(gene_pairs.values()):
    for row, gene in enumerate(genes):
        ax = axes[row, column]
        values = domains[gene]
        ax.scatter(domains.x, domains.y, c=values / values.max(), vmin=0, vmax=1,
                   s=3, linewidths=0, cmap="viridis")
        ax.set_title(gene)
        ax.set_aspect("equal")
        ax.invert_yaxis()
        ax.set_axis_off()
fig.tight_layout()
display(fig)
plt.close(fig)

terms = pd.read_csv(benchmark / "fig04_go_terms.csv")
display(terms[["domain", "GO_ID", "term", "overlap", "q"]])

## Optional current-model retraining

Set `STBSR_RUN_MODE=train` to run the fixed joint model on the original
Stage 57 data at `<data-root>/ARTISTA/Stage57.h5ad`. It must contain the original `layers['counts']` matrix. This evaluates the
two KMeans readouts of the current model for seed 46. The historical
Fig. 4a partition is available in the replay section above.

In [ ]:
if mode == "train":
    spec = config["dataset_specs"][dataset]
    data_path = data_root / spec["path"]
    if not data_path.exists():
        raise FileNotFoundError(f"Set STBSR_DATA_ROOT. Missing: {data_path}")

    outputs = repo / "outputs"
    outputs.mkdir(exist_ok=True)
    out = Path(mkdtemp(prefix=f"{dataset}_seed{seed}_", dir=outputs))
    replay = config.get("benchmark_replay_devices", {}).get(dataset, device)
    if device == "cpu":
        replay = "cpu"
    command = [
        sys.executable, "-m", "STBSR.cli", "train",
        "--dataset", dataset, "--input", str(data_path), "--seed", str(seed),
        "--device", device, "--replay-device", replay, "--out", str(out),
    ]
    subprocess.run(command, cwd=repo, check=True, timeout=config["maximum_case_seconds"] + 180)

    reference = read_reference(dataset, spec, data_path)
    prediction_path = out / "predictions.npz"
    results = pd.DataFrame(evaluate_file(prediction_path, reference)).set_index("readout")
    display(results[["n_scored", "ARI", "NMI", "BF", "BandAcc"]])

    with np.load(prediction_path, allow_pickle=False) as values:
        coords = values["coordinates"]
        frame = pd.DataFrame({"x": coords[:, 0], "y": coords[:, 1]})
        frame["reference"] = reference.reindex(values["barcodes"]).to_numpy()
        replotted = {
            f"{label} (ARI {results.loc[key, 'ARI']:.4f})":
                display_labels(frame["reference"], pd.Series(values[key], index=frame.index))
            for key, label in (("embedding64", "64D"), ("reconstruction_unit", "Reconstruction"))
        }
    fig = plot_domains(frame, replotted, size=spot_size)
    display(fig)
    plt.close(fig)
    print("Training output:", out)
else:
    print("Paper replay completed. Enable training with STBSR_RUN_MODE=train.")